### Name: Se Rang Seo (Simon)
### Assignment 14: Bayes Theorem
### Professor: Dr. Romanowsky
### Course: Phys255 Machine Learning 2026

### Description
The very early universe (around 12–13 billion years ago) was a time when supermassive black holes began growing and emitting radiation at tremendous rates, at the centers of galaxies.  Observationally, these are called quasars or QSOs (quasi-stellar objects), and they can be identified using instruments that detect spectral lines.  However, the lines can be very faint owing to their distance, and can be confused with noise, causing foreground stars to be misclassified as quasars.

Here is an [example quasar candidate](https://skyserver.sdss.org/dr17/VisualTools/explore/summary?objId=1237674649928663641) from the Sloan Digital Sky Survey, which was selected first as a quasar candidate using [photometric techniques including a neural network](http://adsabs.harvard.edu/abs/2012ApJS..199....3R) to an external site., and followed up with a [spectrum](http://adsabs.harvard.edu/abs/2012AJ....144..144B) to an external site. which also classifies the object as a quasar.

### Directions
1. **Use Bayes' theorem to estimate the probability that it really is a quasar.**  As background information: the probability that an object is correctly classified from its spectrum is 80%, and the photometric selection process is reliable at the 63% level.  Hint:  think about the photometric selection as "prior" information.

2. Next, decide if it would be more useful to increase the spectroscopic or photometric reliability by 10%.

3.  
   - **a.** Finally, generate a mock universe of a large number of stars and quasars selected photometrically.  
   - **b.** Then classify them spectroscopically, introducing random errors in the selection and classification.  
   - **c.** Count the numbers of true positives, true negatives, false positives, and false negatives.  
   - **d.** Check the Bayesian calculation above.

# 1.
Calculating the Bayes' theorem.

Bayes' Theorem:

$P(A|B) = \frac{P(B|A)P(A)}{P(B)} = \frac{P(B|A)P(A)}{P(B|A)P(A)+P(B|\neg A)P(\neg A)}$

Let:

A = Photometric selection, object really is a quasar : P(A) = 63% = 0.63
 
B = Spectroscopic classification, spectrum says quasar

P(B∣A)=0.80 (sensitivity)

We want to find: P(A|B) = P(photometrically, actually a quasar given the spectroscopy classifier says 'quasar')

$P(A|B) = \frac{0.8\times 0.63}{(0.8\times0.63+0.2\times0.37)}$

In [128]:
bayes = (0.8*0.63)/(0.8*0.63 + 0.2*0.37)

print(f"Bayes' Theorem: P(A|B) = {bayes*100:.2f}%")

Bayes' Theorem: P(A|B) = 87.20%


# 2.

- a. Increase spectroscopic reliability by 10%

P(A) = 80% = .8

P(B) = 63% + .10% = 73% = .73

In [129]:
bayes_spec_up = (0.8*0.73)/(0.8*0.73 + 0.2*0.27)

print(f"Bayes' Theorem: P(A|B) = {bayes_spec_up*100:.2f}%")

Bayes' Theorem: P(A|B) = 91.54%


- b. Increase photometric reliability by 10%

P(A) = 80% + 10% = 90% = .9

P(B) = 63% = .63

In [130]:
bayes_photo_up = (0.9*0.63)/(0.9*0.63 + 0.1*0.37)

print(f"Bayes' Theorem: P(A|B) = {bayes_photo_up*100:.2f}%")

Bayes' Theorem: P(A|B) = 93.87%


$\text{\color{cyan}It is better to increase the photometric reliability than spectroscopic reliability!}$

# 3.

- a. generating mock universe of many stars and quasars selected photometrics and classifying them spectroscopically.

In [131]:
import numpy as np

Dumping this from ai to do the Monte Carlo simulation

In [132]:
def monte_carlo_bayes(prior_q, acc):
    """
    Monte Carlo simulation to verify Bayes' theorem for quasar classification.
    
    Parameters:
    -----------
    prior_q : float
        Photometric reliability P(A) = probability object is truly a quasar
    acc : float
        Spectroscopic accuracy P(B|A) = sensitivity and specificity (assumed equal)
    n_trials : int
        Number of Monte Carlo trials (default: 10000)
    n_obs : int
        Number of objects per trial (default: 1000)
    
    Returns:
    --------
    dict : Dictionary containing Bayesian prediction and Monte Carlo results
    """
    n_obs=1000
    n_trials=10000
    # Bayesian prediction
    bayes = (acc * prior_q) / (acc * prior_q + (1-acc)*(1-prior_q))
    
    # Monte Carlo simulation
    results = []
    
    for _ in range(n_trials):
        # True state (photometric selection)
        is_q = np.random.rand(n_obs) < prior_q
        
        # Spectroscopic classification
        spec_q = np.zeros(n_obs, dtype=bool)
        
        # Quasars: correct with prob acc
        q_idx = np.where(is_q)[0]
        correct = np.random.rand(len(q_idx)) < acc
        spec_q[q_idx[correct]] = True
        
        # Stars: wrong with prob 1-acc
        s_idx = np.where(~is_q)[0]
        wrong = np.random.rand(len(s_idx)) < (1-acc)
        spec_q[s_idx[wrong]] = True
        
        # Posterior = TP / (TP+FP)
        tp = np.sum(is_q & spec_q)
        fp = np.sum(~is_q & spec_q)
        results.append(tp / (tp+fp) if (tp+fp) > 0 else 0)
    
    # Results
    mean_mc = np.mean(results)
    std_mc = np.std(results)
    
    return {
        'bayesian': bayes,
        'monte_carlo_mean': mean_mc,
        'monte_carlo_std': std_mc,
        'difference': mean_mc - bayes,
        'all_results': results
    }

In [133]:
prior_q0 = 0.63
acc0 = 0.8

result00 = monte_carlo_bayes(prior_q0, acc0)

print(f"P(A) = {prior_q0} and P(B|A) = {acc0}")
print(f"Bayesian Prediction: {result00['bayesian']*100:.2f}%")
print(f"Monte Carlo Mean: {result00['monte_carlo_mean']*100:.2f}%")

P(A) = 0.63 and P(B|A) = 0.8
Bayesian Prediction: 87.20%
Monte Carlo Mean: 87.17%


In [134]:
prior_q1 = 0.73

result10 = monte_carlo_bayes(prior_q1, acc0)
    
print(f"P(A) = {prior_q1} and P(B|A) = {acc0}")
print(f"Bayesian Prediction: {result10['bayesian']*100:.2f}%")
print(f"Monte Carlo Mean: {result10['monte_carlo_mean']*100:.2f}%")

P(A) = 0.73 and P(B|A) = 0.8
Bayesian Prediction: 91.54%
Monte Carlo Mean: 91.53%


In [135]:
acc1 = 0.9

result01 = monte_carlo_bayes(prior_q0, acc1)

print(f"P(A) = {prior_q0} and P(B|A) = {acc1}")
print(f"Bayesian Prediction: {result01['bayesian']*100:.2f}%")
print(f"Monte Carlo Mean: {result01['monte_carlo_mean']*100:.2f}%")

P(A) = 0.63 and P(B|A) = 0.9
Bayesian Prediction: 93.87%
Monte Carlo Mean: 93.86%


- b. Note the random errors form selections and classification.


In [136]:
print(f"P(A) = {prior_q0} and P(B|A) = {acc0}")
print(f"Monte Carlo Std Dev: {result00['monte_carlo_std']*100:.2f}%\n")


print(f"P(A) = {prior_q1} and P(B|A) = {acc0}")
print(f"Monte Carlo Std Dev: {result10['monte_carlo_std']*100:.2f}%\n")


print(f"P(A) = {prior_q0} and P(B|A) = {acc1}")
print(f"Monte Carlo Std Dev: {result01['monte_carlo_std']*100:.2f}%")

P(A) = 0.63 and P(B|A) = 0.8
Monte Carlo Std Dev: 1.38%

P(A) = 0.73 and P(B|A) = 0.8
Monte Carlo Std Dev: 1.12%

P(A) = 0.63 and P(B|A) = 0.9
Monte Carlo Std Dev: 0.97%


- c. Identify true positives, true negatives, false positives, false negatives, and perform Bayes theorem.

In [137]:
def make_confusion_matrix(prior_q, acc, n_obs=1000, n_runs=100):
    """Returns average 2x2 confusion matrix and average precision over n_runs"""
    total_TP = 0
    total_FP = 0
    total_FN = 0
    total_TN = 0
    precisions = []
    
    for _ in range(n_runs):
        is_q = np.random.rand(n_obs) < prior_q
        spec_q = np.zeros(n_obs, dtype=bool)
        
        q_idx = np.where(is_q)[0]
        correct = np.random.rand(len(q_idx)) < acc
        spec_q[q_idx[correct]] = True
        
        s_idx = np.where(~is_q)[0]
        wrong = np.random.rand(len(s_idx)) < (1-acc)
        spec_q[s_idx[wrong]] = True
        
        TP = np.sum(is_q & spec_q)
        FP = np.sum(~is_q & spec_q)
        FN = np.sum(is_q & ~spec_q)
        TN = np.sum(~is_q & ~spec_q)
        
        total_TP += TP
        total_FP += FP
        total_FN += FN
        total_TN += TN
        precisions.append(TP / (TP + FP))
    
    avg_matrix = np.array([[total_TP/n_runs, total_FP/n_runs], 
                           [total_FN/n_runs, total_TN/n_runs]])
    
    return avg_matrix, np.mean(precisions)

In [138]:
matrix00, precision00 = make_confusion_matrix(0.63, 0.80)

print(f"P(A) = {prior_q0} and P(B|A) = {acc0}")
print(matrix00)
print(f"Confusion Matrix Precision: {precision00*100:.2f}%")

P(A) = 0.63 and P(B|A) = 0.8
[[504.67  74.32]
 [125.2  295.81]]
Confusion Matrix Precision: 87.16%


In [139]:
matrix10, precision10 = make_confusion_matrix(0.73, 0.80)

print(f"P(A) = {prior_q1} and P(B|A) = {acc0}")
print(matrix10)
print(f"Confusion Matrix Precision: {precision10*100:.2f}%")

P(A) = 0.73 and P(B|A) = 0.8
[[585.69  52.16]
 [145.81 216.34]]
Confusion Matrix Precision: 91.82%


In [140]:
# Usage
matrix01, precision01 = make_confusion_matrix(0.63, 0.90)

print(f"P(A) = {prior_q0} and P(B|A) = {acc1}")
print(matrix01)
print(f"Confusion Matrix Precision: {precision01*100:.2f}%")

P(A) = 0.63 and P(B|A) = 0.9
[[569.16  36.37]
 [ 63.29 331.18]]
Confusion Matrix Precision: 93.99%


# Conclusion

### $\textcolor{cyan}{P(A) = 0.63 \quad \text{and} \quad P(B|A) = 0.8}$
$
\textcolor{cyan}{
\begin{aligned}
&\text{Bayesian Prediction: 87.20\%} \\
&\text{Monte Carlo Mean: 87.19\% } \pm \text{ 1.39\%} \\
&\text{Confusion Matrix Precision: 87.11\%}
\end{aligned}
}
$

### $\textcolor{orange}{P(A) = 0.73 \quad \text{and} \quad P(B|A) = 0.8}$

$
\textcolor{orange}{
\begin{aligned}
&\text{Bayesian Prediction: 91.54\%} \\
&\text{Monte Carlo Mean: 91.54\%} \pm \text{1.10\%} \\
&\text{Confusion Matrix Precision: 91.59\%}
\end{aligned}
}
$

### $\textcolor{lightgreen}{P(A) = 0.63 \quad \text{and} \quad P(B|A) = 0.9}$

$
\textcolor{lightgreen}{
\begin{aligned}
&\text{Bayesian Prediction: 93.87\%} \\
&\text{Monte Carlo Mean: 93.87\%} \pm \text{0.97\%} \\
&\text{Confusion Matrix Precision: 93.87\%}
\end{aligned}
}
$

What did I learn?

Bayes' Theorem is a conditional probability that measures precision which relates back to the confusion matrix and the true and false values.

In retrospective point of view, confusion matrix can be used to measure conditional probability. -mindblown-

(I remember there being a square/table method to solve conditonal probabiltiies when I was tutoring some students in algebra 2 and pre-calculus.
Unknowingly, maybe I was working with a confusion matrix.) [video](https://www.youtube.com/watch?v=sqDVrXq_eh0) - "Contingency table" ; Confusion Matrix is a restricted type of contingency table; [paper](https://analyticalsciencejournals.onlinelibrary.wiley.com/doi/pdf/10.1002/cem.3331)

Enough tangent, onto the next assignment..